<h1 align="center">LAB1S1_p4. Contextual word-embeddings for text representation</h1>

<h3 style="display:block; margin-top:5px;" align="center">Natural Language and Information Retrieval</h3>
<h3 style="display:block; margin-top:5px;" align="center">Degree in Data Science</h3>
<h3 style="display:block; margin-top:5px;" align="center">2024-2025</h3>    
<h3 style="display:block; margin-top:5px;" align="center">ETSInf. Universitat Politècnica de València</h3>
<br>

### Put your names here

- Bartosz Stoklosa
- Baranyi Sándor

In [3]:
!pip install -U transformers
!pip install -U emoji
!pip install -U ipywidgets
!pip install -U torch


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached setuptools-76.0.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   - -------------------------------------- 6.3/204.1 MB 38.6 MB/s eta 0:00:06
   --- ------------------------------------ 17.0/204.1 MB 44.3 MB/s eta 0:00:05
   ----- ---------------------------------- 27.5/204.1 MB 47.0 MB/s eta 0:00:04
   ------- -------------------------------- 39.8/204.1 MB 49.7 MB/s eta 0:00:04
   --------- ------------------------------ 47.2/204.1 MB 46.9 MB/s eta 0:00:04
   ----------- ---------------------------- 59.5/204.1 MB 48.8 MB/s eta 0:00:03
   ------------- -------------------------- 69.5/204.1 MB 48.5 MB/s eta 0:00:03
   --------------- ------------------------ 79.4/204.1 MB 48.1 MB/s eta 0:00:03
   ----------------- ---------------------- 87.3/204.1 MB 47.1 MB/s eta 0:00:03
   ------------------- -------------------- 97.0/204.1 MB 46.8 MB/s eta 0:00:03


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Some libraries

In [2]:
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from transformers import BertTokenizer, BertModel, RobertaTokenizer, RobertaModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

C:\Users\Sanyi\Desktop\Iskola\Órák\6.Félév\UPV\Natural_Language_And_Information_Retrieval\Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Read the corpora

In [3]:
filepath = {
    "english": "EXIST2024_EN_examples_mini.csv",
    "spanish": "EXIST2024_ES_examples_mini.csv"
}
df = {k: pd.read_csv(v, sep="\t") for k, v in filepath.items()}

# OR USING Google colab
#from google.colab import drive
#drive.mount('/content/drive')
#df = {
#    "english": pd.read_csv("/content/drive/MyDrive/LNR/LNR2025/Lab1/EXIST2024_EN_examples.csv", sep="\t"),
#    "spanish": pd.read_csv("/content/drive/MyDrive/LNR/LNR2025/Lab1/EXIST2024_ES_examples.csv", sep="\t")
#}


## Model names

In [4]:
modelnames = {
    "english": ["bert-base-uncased", "roberta-base"],
    "spanish": ["dccuchile/bert-base-spanish-wwm-uncased", "PlanTL-GOB-ES/roberta-base-bne"]
}

## Which device to use?

In [5]:
if torch.backends.mps.is_available():  # Mac M? GPU
    device = torch.device("mps")
elif torch.cuda.is_available():  # Nvidia GPU
    device = torch.device("cuda")
else:  # CPU
    device = torch.device("cpu")
print(device)

cpu


## Load the tokenizers and the models

In [6]:
tokenizers = {}
models = {}

for lang, models_list in modelnames.items():
    tokenizers[lang] = [AutoTokenizer.from_pretrained(model) for model in models_list]
    models[lang] = [AutoModel.from_pretrained(model).to(device) for model in models_list]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-bne and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Compute tweets representations

In [7]:
batch_size = 16 

def get_cls_embeddings(texts, tokenizer, model):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

embeddings = {}

for lang in df:
    embeddings[lang] = {}
    for i, model_name in enumerate(modelnames[lang]):
        embeddings[lang][model_name] = []
        for start in range(0, len(df[lang]), batch_size):
            batch = df[lang]['text'][start:start + batch_size].tolist()
            cls_embeddings = get_cls_embeddings(batch, tokenizers[lang][i], models[lang][i])
            embeddings[lang][model_name].append(cls_embeddings)
        embeddings[lang][model_name] = np.vstack(embeddings[lang][model_name])

## Compute cosine similarities

In [12]:
def find_most_similar_pairs(embeddings, labels):
    most_similar_pairs = {}
    for label in set(labels):
        indices = [i for i, l in enumerate(labels) if l == label]
        label_embeddings = embeddings[indices]
        similarities = cosine_similarity(label_embeddings)
        np.fill_diagonal(similarities, -1)
        max_sim_idx = np.unravel_index(np.argmax(similarities), similarities.shape)
        most_similar_pairs[label] = (indices[max_sim_idx[0]], indices[max_sim_idx[1]])
    return most_similar_pairs


## Show results

In [13]:
for lang in df:
    for model_name in modelnames[lang]:
        print(f"Model: {model_name}")
        most_similar_pairs = find_most_similar_pairs(embeddings[lang][model_name], df[lang]['label'])
        for label, pair in most_similar_pairs.items():
            print(f"label: {label}")
            print(f"sentence1: {df[lang]['text'][pair[0]]}")
            print(f"sentence2: {df[lang]['text'][pair[1]]}")
            print(f"distance: {cosine_similarity([embeddings[lang][model_name][pair[0]]], [embeddings[lang][model_name][pair[1]]])[0][0]:.4f}")
            print("\n")

Model: bert-base-uncased
label: YES
sentence1: The mighty ass. Call me sexist I do not care. https://t.co/LzXw4iRbLR
sentence2: @RP_JetBlack Not shaming you at all! I too am a massive slut and a total cock tease. https://t.co/HbZiZXRi0N
distance: 0.9774


label: NO
sentence1: I still wish they turned this into a boss fight. https://t.co/HyvPYJPHJc
sentence2: I don't particularly care or want to know about the cock carousel. Everyone has a past. https://t.co/73WMTyEKHt
distance: 0.9739


Model: roberta-base
label: YES
sentence1: @lkmeenha we can’t even have a day without women making it about themselves 🙄
sentence2: @BigDILF01 Can’t go a day without women womening
distance: 0.9990


label: NO
sentence1: Thank you beautiful friend 😊Sending love and 🕯️🚨 light your way 💓 https://t.co/EbPpAKWqjo https://t.co/n3MDADAH7N
sentence2: Have a lovely day beautiful sunshine 🌞 ❤️♥️💜🔥🔥🔥🔥🔥🔥🐎 https://t.co/w4yoltPn6z https://t.co/qDf358MMsH
distance: 0.9992


Model: dccuchile/bert-base-spanish-wwm-uncas